In [3]:
# # =====================================================
# # Walk-Forward Day-Ahead Quantile XGBoost - Enhanced Version
# # Electricity Market - Declared Power
# # =====================================================

# import pandas as pd
# import numpy as np
# import xgboost as xgb
# from sklearn.model_selection import TimeSeriesSplit
# import warnings
# warnings.filterwarnings('ignore')

# # =========================
# # تنظیمات
# # =========================
# INPUT_FILE = "merged_output2.csv"
# OUTPUT_FILE = "xgboost5_enhanced.xlsx"

# TARGET = "POWER"
# DATE_COL = "DATE_MILADI"
# HOUR_COL = "HOUR"
# EBRAZ_COL = "ebraz"

# QUANTILE = 1.0        # محافظه‌کار ولی نه سقوط‌کننده
# MIN_RATIO = 1.0        # کف منطقی نسبت به lag_24

# # =========================
# # خواندن داده
# # =========================
# print("خواندن داده...")
# df = pd.read_csv(INPUT_FILE)
# df[DATE_COL] = pd.to_datetime(df[DATE_COL])
# df = df.sort_values([DATE_COL, HOUR_COL]).reset_index(drop=True)

# # =========================
# # ویژگی‌های زمانی پایه
# # =========================
# df["hour"] = df[HOUR_COL]
# df["dayofweek"] = df[DATE_COL].dt.dayofweek
# df["month"] = df[DATE_COL].dt.month
# df["day"] = df[DATE_COL].dt.day
# df["quarter"] = df[DATE_COL].dt.quarter

# # =========================
# # Lagها
# # =========================
# print("ایجاد Lagها...")
# df["lag_24"] = df[TARGET].shift(24)
# df["lag_48"] = df[TARGET].shift(48)
# df["lag_72"] = df[TARGET].shift(72)
# df["lag_168"] = df[TARGET].shift(168)  # هفته قبل

# # =========================
# # ویژگی‌های پیشرفته
# # =========================
# print("ایجاد ویژگی‌های پیشرفته...")

# # میانگین متحرک
# df["MA_24"] = df[TARGET].rolling(24, min_periods=1).mean()
# df["MA_168"] = df[TARGET].rolling(168, min_periods=1).mean()

# # تغییرات
# df["delta_24"] = df["lag_24"] - df["lag_48"]
# df["delta_48"] = df["lag_48"] - df["lag_72"]

# # ویژگی‌های تعاملی
# df["lag24_hour"] = df["lag_24"] * df["hour"]
# df["hour_dayofweek"] = df["hour"] * df["dayofweek"]

# # ویژگی‌های فصلی
# df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
# df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
# df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
# df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# # ویژگی‌های منطقی
# df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
# df["is_night"] = ((df["hour"] >= 0) & (df["hour"] <= 5)).astype(int)
# df["is_peak"] = ((df["hour"] >= 17) & (df["hour"] <= 21)).astype(int)

# # نسبت‌ها
# df["ratio_24_48"] = df["lag_24"] / (df["lag_48"] + 1)
# df["ratio_24_ma168"] = df["lag_24"] / (df["MA_168"] + 1)

# # =========================
# # حذف مقادیر NaN
# # =========================
# initial_rows = len(df)
# df = df.dropna().reset_index(drop=True)
# print(f"حذف {initial_rows - len(df)} سطر با مقادیر NaN")

# # =========================
# # لیست ویژگی‌ها
# # =========================
# FEATURES = [
#     # پایه
#     "hour", "dayofweek", "month", "quarter",
    
#     # Lagها
#     "lag_24", "lag_48", "lag_72", "lag_168",
    
#     # میانگین متحرک
#     "MA_24", "MA_168",
    
#     # تغییرات
#     "delta_24", "delta_48",
    
#     # ویژگی‌های خارجی (فرض می‌شود در داده موجودند)
#     "DAMA", "ROTOOBAT",
    
#     # ویژگی‌های فصلی
#     "hour_sin", "hour_cos", "month_sin", "month_cos",
    
#     # ویژگی‌های منطقی
#     "is_weekend", "is_night", "is_peak",
    
#     # نسبت‌ها
#     "ratio_24_48", "ratio_24_ma168",
    
#     # ویژگی‌های تعاملی
#     "lag24_hour", "hour_dayofweek"
# ]

# print(f"تعداد ویژگی‌ها: {len(FEATURES)}")

# # =========================
# # Quantile Loss
# # =========================
# def quantile_objective(alpha):
#     def loss(y_true, y_pred):
#         grad = np.where(y_pred < y_true, -alpha, 1 - alpha)
#         hess = np.ones_like(grad)
#         return grad, hess
#     return loss

# # =========================
# # تابع تنظیم بازار هوشمند
# # =========================
# def smart_market_adjustment(preds, context_df):
#     """
#     تنظیم هوشمند پیش‌بینی‌ها با منطق بازار
#     """
#     adjusted_preds = preds.copy()
    
#     # 1. کف پویا بر اساس lag_24
#     lag24 = context_df["lag_24"].values
#     floor = lag24 * MIN_RATIO
    
#     # 2. محافظه‌کاری بیشتر در ساعات پیک
#     is_peak = context_df["is_peak"].values
#     peak_factor = np.where(is_peak, 0.95, 1.0)  # 5% محافظه‌کارتر در پیک
#     adjusted_preds = adjusted_preds * peak_factor
    
#     # 3. اعمال کف
#     adjusted_preds = np.maximum(adjusted_preds, floor)
    
#     # 4. سقف منطقی بر اساس حداکثر تاریخی
#     if 'historical_max' in context_df.columns:
#         ceiling = context_df['historical_max'].values * 1.25  # 25% بالاتر
#         adjusted_preds = np.minimum(adjusted_preds, ceiling)
    
#     # 5. تنظیم برای ابراز = 0
#     if EBRAZ_COL in context_df.columns:
#         adjusted_preds[context_df[EBRAZ_COL].values == 0] = 0
    
#     # 6. اطمینان از مقادیر مثبت
#     adjusted_preds = np.maximum(adjusted_preds, 0)
    
#     return adjusted_preds

# # =========================
# # تابع محاسبه حداکثر تاریخی
# # =========================
# def calculate_historical_max(df, date_col, target_col):
#     """
#     محاسبه حداکثر تاریخی برای هر ساعت
#     """
#     df['hour'] = df[DATE_COL].dt.hour
#     historical_max = df.groupby('hour')[target_col].max().reset_index()
#     historical_max.columns = ['hour', 'historical_max']
#     return historical_max

# # =========================
# # محاسبه حداکثر تاریخی
# # =========================
# historical_max_df = calculate_historical_max(df, DATE_COL, TARGET)

# # =========================
# # Ensemble Model
# # =========================
# class QuantileEnsemble:
#     def __init__(self, quantiles=[0.85, 0.92, 1.0], weights=None):
#         self.quantiles = quantiles
#         self.weights = weights if weights else [0.2, 0.3, 0.5]
#         self.models = []
        
#     def fit(self, X_train, y_train):
#         self.models = []
#         for q in self.quantiles:
#             model = xgb.XGBRegressor(
#                 n_estimators=400,
#                 max_depth=5,
#                 learning_rate=0.03,
#                 subsample=0.75,
#                 colsample_bytree=0.7,
#                 min_child_weight=3,
#                 reg_alpha=0.05,
#                 reg_lambda=1.0,
#                 objective=quantile_objective(q),
#                 random_state=42 + int(q*100),
#                 n_jobs=-1,
#                 verbosity=0
#             )
#             model.fit(X_train, y_train)
#             self.models.append(model)
#         return self
    
#     def predict(self, X_test):
#         predictions = []
#         for i, model in enumerate(self.models):
#             pred = model.predict(X_test)
#             predictions.append(pred)
        
#         # ترکیب وزنی
#         ensemble_pred = np.zeros_like(predictions[0])
#         for i, (pred, weight) in enumerate(zip(predictions, self.weights)):
#             ensemble_pred += pred * weight
        
#         return ensemble_pred, predictions  # بازگشت پیش‌بینی انسامبل و تک‌تک مدل‌ها

# # =========================
# # Walk Forward Forecast
# # =========================
# print("\nشروع پیش‌بینی Walk-Forward...")

# df["DECLARED"] = np.nan
# df["CONFIDENCE"] = np.nan
# df["ENSEMBLE_VAR"] = np.nan  # واریانس پیش‌بینی‌های انسامبل

# unique_days = sorted(df[DATE_COL].dt.date.unique())
# total_days = len(unique_days)

# # حداقل 30 روز برای آموزش
# start_day = 30

# for day_idx in range(start_day, total_days - 1):
#     train_days = unique_days[:day_idx]
#     predict_day = unique_days[day_idx]
    
#     train_idx = df[DATE_COL].dt.date.isin(train_days)
#     test_idx = df[DATE_COL].dt.date == predict_day
    
#     if not test_idx.any():
#         continue
    
#     X_train = df.loc[train_idx, FEATURES]
#     y_train = df.loc[train_idx, TARGET]
#     X_test = df.loc[test_idx, FEATURES]
    
#     # آموزش مدل انسامبل
#     ensemble = QuantileEnsemble()
#     ensemble.fit(X_train, y_train)
    
#     # پیش‌بینی
#     ensemble_pred, individual_preds = ensemble.predict(X_test)
    
#     # محاسبه واریانس برای اطمینان
#     pred_array = np.array(individual_preds)
#     ensemble_variance = np.var(pred_array, axis=0)
    
#     # تنظیمات بازار
#     context_df = df.loc[test_idx].copy()
#     context_df = context_df.merge(historical_max_df, on='hour', how='left')
    
#     adjusted_preds = smart_market_adjustment(ensemble_pred, context_df)
    
#     # ذخیره نتایج
#     df.loc[test_idx, "DECLARED"] = adjusted_preds
#     df.loc[test_idx, "ENSEMBLE_VAR"] = ensemble_variance
    
#     # محاسبه اطمینان (معکوس واریانس نرمال‌شده)
#     if ensemble_variance.max() > 0:
#         confidence = 1 - (ensemble_variance / ensemble_variance.max())
#         df.loc[test_idx, "CONFIDENCE"] = confidence
    
#     if day_idx % 30 == 0:
#         print(f"پردازش روز {day_idx + 1} از {total_days}")

# # =========================
# # منطق نهایی بازار
# # =========================
# print("\nاعمال منطق نهایی بازار...")

# # تنظیم برای ابراز = 0
# if EBRAZ_COL in df.columns:
#     df.loc[df[EBRAZ_COL] == 0, "DECLARED"] = 0

# # اطمینان از مقادیر مثبت
# df["DECLARED"] = df["DECLARED"].clip(lower=0)

# # گرد کردن به عدد صحیح
# df["DECLARED"] = df["DECLARED"].round(2)

# # =========================
# # ارزیابی
# # =========================
# print("\n" + "="*50)
# print("ارزیابی نتایج:")
# print("="*50)

# # محاسبه خطا
# err = df["DECLARED"] - df[TARGET]
# valid_err = err[~np.isnan(err)]

# if len(valid_err) > 0:
#     # خطاهای مثبت (Over-prediction)
#     over_err = valid_err[valid_err > 0]
#     # خطاهای منفی (Under-prediction)
#     under_err = valid_err[valid_err < 0]
    
#     if len(over_err) > 0:
#         mae_pos = np.mean(np.abs(over_err))
#         mape_pos = np.mean(np.abs(over_err / df.loc[over_err.index, TARGET])) * 100
#     else:
#         mae_pos = 0
#         mape_pos = 0
    
#     if len(under_err) > 0:
#         mae_neg = np.mean(np.abs(under_err))
#         mape_neg = np.mean(np.abs(under_err / df.loc[under_err.index, TARGET])) * 100
#     else:
#         mae_neg = 0
#         mape_neg = 0
    
#     # محاسبه امتیاز بازار
#     market_score = 5 * mae_pos + mae_neg
    
#     # دقت کلی
#     total_mae = np.mean(np.abs(valid_err))
#     total_mape = np.mean(np.abs(valid_err / df.loc[valid_err.index, TARGET])) * 100
    
#     print(f"MAE Positive (Over-prediction): {mae_pos:.2f} ({mape_pos:.1f}%)")
#     print(f"MAE Negative (Under-prediction): {mae_neg:.2f} ({mape_neg:.1f}%)")
#     print(f"Total MAE: {total_mae:.2f} ({total_mape:.1f}%)")
#     print(f"Market Score (5*Over + Under): {market_score:.2f}")
#     print(f"تعداد پیش‌بینی‌ها: {len(valid_err)}")
    
#     # توزیع خطا
#     print(f"\nتوزیع خطا:")
#     print(f"  Over-prediction: {len(over_err)} ({len(over_err)/len(valid_err)*100:.1f}%)")
#     print(f"  Under-prediction: {len(under_err)} ({len(under_err)/len(valid_err)*100:.1f}%)")
#     print(f"  Exact: {sum(valid_err == 0)} ({sum(valid_err == 0)/len(valid_err)*100:.1f}%)")
# else:
#     print("هیچ داده معتبری برای ارزیابی وجود ندارد")

# # =========================
# # ویژگی‌های مهم
# # =========================
# if len(unique_days) > start_day:
#     # آموزش مدل نهایی روی همه داده‌های آموزشی
#     last_train_idx = df[DATE_COL].dt.date < unique_days[-1]
#     X_final_train = df.loc[last_train_idx, FEATURES]
#     y_final_train = df.loc[last_train_idx, TARGET]
    
#     final_model = xgb.XGBRegressor(
#         n_estimators=300,
#         max_depth=5,
#         learning_rate=0.05,
#         objective=quantile_objective(QUANTILE),
#         random_state=42
#     )
#     final_model.fit(X_final_train, y_final_train)
    
#     # اهمیت ویژگی‌ها
#     feature_importance = pd.DataFrame({
#         'feature': FEATURES,
#         'importance': final_model.feature_importances_
#     }).sort_values('importance', ascending=False)
    
#     print("\n" + "="*50)
#     print("۱۰ ویژگی مهم برتر:")
#     print("="*50)
#     for i, row in feature_importance.head(10).iterrows():
#         print(f"{row['feature']}: {row['importance']:.4f}")

# # =========================
# # ذخیره نتایج
# # =========================
# print(f"\nذخیره نتایج در {OUTPUT_FILE}...")

# # ایجاد DataFrame خلاصه نتایج
# results_summary = pd.DataFrame({
#     'تاریخ': df[DATE_COL],
#     'ساعت': df[HOUR_COL],
#     'قدرت واقعی': df[TARGET],
#     'قدرت اعلان شده': df["DECLARED"],
#     'خطا': err,
#     'نوع خطا': pd.cut(err, 
#                       bins=[-np.inf, -0.01, 0.01, np.inf], 
#                       labels=['Under', 'Exact', 'Over']),
#     'اطمینان': df["CONFIDENCE"],
#     'واریانس انسامبل': df["ENSEMBLE_VAR"],
#     'Lag_24': df["lag_24"]
# })

# # اضافه کردن ویژگی‌های مهم به عنوان ستون‌های اضافی
# for feature in FEATURES[:5]:  # 5 ویژگی اول
#     results_summary[feature] = df[feature]

# # ذخیره در اکسل با چندین sheet
# with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
#     results_summary.to_excel(writer, sheet_name='نتایج', index=False)
    
#     # sheet دوم: آمارهای خلاصه
#     summary_stats = pd.DataFrame({
#         'متریک': ['MAE Positive', 'MAE Negative', 'Total MAE', 'Market Score', 
#                   'Over-prediction %', 'Under-prediction %', 'Exact %'],
#         'مقدار': [f"{mae_pos:.2f}", f"{mae_neg:.2f}", f"{total_mae:.2f}", 
#                   f"{market_score:.2f}", 
#                   f"{len(over_err)/len(valid_err)*100:.1f}%" if len(valid_err) > 0 else "N/A",
#                   f"{len(under_err)/len(valid_err)*100:.1f}%" if len(valid_err) > 0 else "N/A",
#                   f"{sum(valid_err == 0)/len(valid_err)*100:.1f}%" if len(valid_err) > 0 else "N/A"]
#     })
#     summary_stats.to_excel(writer, sheet_name='آمار', index=False)
    
#     # sheet سوم: اهمیت ویژگی‌ها
#     if 'feature_importance' in locals():
#         feature_importance.to_excel(writer, sheet_name='اهمیت ویژگی‌ها', index=False)

# print("ذخیره‌سازی با موفقیت انجام شد!")
# print(f"\nفایل خروجی: {OUTPUT_FILE}")
# print("شامل سه Sheet:")
# print("  1. نتایج: تمام پیش‌بینی‌ها و خطاها")
# print("  2. آمار: خلاصه عملکرد مدل")
# print("  3. اهمیت ویژگی‌ها: مهم‌ترین ویژگی‌های مدل")

In [1]:
# =====================================================
# Walk-Forward Day-Ahead Quantile XGBoost - Enhanced Version
# Electricity Market - Declared Power
# =====================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# =========================
# تنظیمات
# =========================
INPUT_FILE = "merged_output2.csv"
OUTPUT_FILE = "xgboost5_enhanced.xlsx"

TARGET = "POWER"
DATE_COL = "DATE_MILADI"
HOUR_COL = "HOUR"
EBRAZ_COL = "ebraz"

QUANTILE = 1.0        # محافظه‌کار ولی نه سقوط‌کننده
MIN_RATIO = 1.0        # کف منطقی نسبت به lag_24

# =========================
# خواندن داده
# =========================
print("خواندن داده...")
df = pd.read_csv(INPUT_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([DATE_COL, HOUR_COL]).reset_index(drop=True)

# =========================
# ویژگی‌های زمانی پایه
# =========================
df["hour"] = df[HOUR_COL]
df["dayofweek"] = df[DATE_COL].dt.dayofweek
df["month"] = df[DATE_COL].dt.month
df["day"] = df[DATE_COL].dt.day
df["quarter"] = df[DATE_COL].dt.quarter

# =========================
# Lagها
# =========================
print("ایجاد Lagها...")
df["lag_24"] = df[TARGET].shift(24)
df["lag_48"] = df[TARGET].shift(48)
df["lag_72"] = df[TARGET].shift(72)
df["lag_168"] = df[TARGET].shift(168)  # هفته قبل

# =========================
# ویژگی‌های پیشرفته
# =========================
print("ایجاد ویژگی‌های پیشرفته...")

# میانگین متحرک
df["MA_24"] = df[TARGET].rolling(24, min_periods=1).mean()
df["MA_168"] = df[TARGET].rolling(168, min_periods=1).mean()

# تغییرات
df["delta_24"] = df["lag_24"] - df["lag_48"]
df["delta_48"] = df["lag_48"] - df["lag_72"]

# ویژگی‌های تعاملی
df["lag24_hour"] = df["lag_24"] * df["hour"]
df["hour_dayofweek"] = df["hour"] * df["dayofweek"]

# ویژگی‌های فصلی
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# ویژگی‌های منطقی
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
df["is_night"] = ((df["hour"] >= 0) & (df["hour"] <= 5)).astype(int)
df["is_peak"] = ((df["hour"] >= 17) & (df["hour"] <= 21)).astype(int)

# نسبت‌ها
df["ratio_24_48"] = df["lag_24"] / (df["lag_48"] + 1)
df["ratio_24_ma168"] = df["lag_24"] / (df["MA_168"] + 1)

# =========================
# حذف مقادیر NaN
# =========================
initial_rows = len(df)
df = df.dropna().reset_index(drop=True)
print(f"حذف {initial_rows - len(df)} سطر با مقادیر NaN")

# =========================
# لیست ویژگی‌ها
# =========================
FEATURES = [
    # پایه
    "hour", "dayofweek", "month", "quarter",
    
    # Lagها
    "lag_24", "lag_48", "lag_72", "lag_168",
    
    # میانگین متحرک
    "MA_24", "MA_168",
    
    # تغییرات
    "delta_24", "delta_48",
    
    # ویژگی‌های خارجی (فرض می‌شود در داده موجودند)
    "DAMA", "ROTOOBAT",
    
    # ویژگی‌های فصلی
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    
    # ویژگی‌های منطقی
    "is_weekend", "is_night", "is_peak",
    
    # نسبت‌ها
    "ratio_24_48", "ratio_24_ma168",
    
    # ویژگی‌های تعاملی
    "lag24_hour", "hour_dayofweek"
]

print(f"تعداد ویژگی‌ها: {len(FEATURES)}")

# =========================
# Quantile Loss
# =========================
def quantile_objective(alpha):
    def loss(y_true, y_pred):
        grad = np.where(y_pred < y_true, -alpha, 1 - alpha)
        hess = np.ones_like(grad)
        return grad, hess
    return loss

# =========================
# تابع تنظیم بازار هوشمند
# =========================
def smart_market_adjustment(preds, context_df):
    """
    تنظیم هوشمند پیش‌بینی‌ها با منطق بازار
    """
    adjusted_preds = preds.copy()
    
    # 1. کف پویا بر اساس lag_24
    lag24 = context_df["lag_24"].values
    floor = lag24 * MIN_RATIO
    
    # 2. محافظه‌کاری بیشتر در ساعات پیک
    is_peak = context_df["is_peak"].values
    peak_factor = np.where(is_peak, 0.95, 1.0)  # 5% محافظه‌کارتر در پیک
    adjusted_preds = adjusted_preds * peak_factor
    
    # 3. اعمال کف
    adjusted_preds = np.maximum(adjusted_preds, floor)
    
    # 4. سقف منطقی بر اساس حداکثر تاریخی
    if 'historical_max' in context_df.columns:
        ceiling = context_df['historical_max'].values * 1.25  # 25% بالاتر
        adjusted_preds = np.minimum(adjusted_preds, ceiling)
    
    # 5. تنظیم برای ابراز = 0
    if EBRAZ_COL in context_df.columns:
        adjusted_preds[context_df[EBRAZ_COL].values == 0] = 0
    
    # 6. اطمینان از مقادیر مثبت
    adjusted_preds = np.maximum(adjusted_preds, 0)
    
    return adjusted_preds

# =========================
# تابع محاسبه حداکثر تاریخی
# =========================
def calculate_historical_max(df, date_col, target_col):
    """
    محاسبه حداکثر تاریخی برای هر ساعت
    """
    df['hour'] = df[DATE_COL].dt.hour
    historical_max = df.groupby('hour')[target_col].max().reset_index()
    historical_max.columns = ['hour', 'historical_max']
    return historical_max

# =========================
# محاسبه حداکثر تاریخی
# =========================
historical_max_df = calculate_historical_max(df, DATE_COL, TARGET)

# =========================
# Ensemble Model
# =========================
class QuantileEnsemble:
    def __init__(self, quantiles=[0.85, 0.92, 1.0], weights=None):
        self.quantiles = quantiles
        self.weights = weights if weights else [0.2, 0.3, 0.5]
        self.models = []
        
    def fit(self, X_train, y_train):
        self.models = []
        for q in self.quantiles:
            model = xgb.XGBRegressor(
                n_estimators=400,
                max_depth=5,
                learning_rate=0.03,
                subsample=0.75,
                colsample_bytree=0.7,
                min_child_weight=3,
                reg_alpha=0.05,
                reg_lambda=1.0,
                objective=quantile_objective(q),
                random_state=42 + int(q*100),
                n_jobs=-1,
                verbosity=0
            )
            model.fit(X_train, y_train)
            self.models.append(model)
        return self
    
    def predict(self, X_test):
        predictions = []
        for i, model in enumerate(self.models):
            pred = model.predict(X_test)
            predictions.append(pred)
        
        # ترکیب وزنی
        ensemble_pred = np.zeros_like(predictions[0])
        for i, (pred, weight) in enumerate(zip(predictions, self.weights)):
            ensemble_pred += pred * weight
        
        return ensemble_pred, predictions  # بازگشت پیش‌بینی انسامبل و تک‌تک مدل‌ها

# =========================
# Walk Forward Forecast
# =========================
print("\nشروع پیش‌بینی Walk-Forward...")

df["DECLARED"] = np.nan
df["CONFIDENCE"] = np.nan
df["ENSEMBLE_VAR"] = np.nan  # واریانس پیش‌بینی‌های انسامبل

unique_days = sorted(df[DATE_COL].dt.date.unique())
total_days = len(unique_days)

# حداقل 30 روز برای آموزش
start_day = 30

for day_idx in range(start_day, total_days - 1):
    train_days = unique_days[:day_idx]
    predict_day = unique_days[day_idx]
    
    train_idx = df[DATE_COL].dt.date.isin(train_days)
    test_idx = df[DATE_COL].dt.date == predict_day
    
    if not test_idx.any():
        continue
    
    X_train = df.loc[train_idx, FEATURES]
    y_train = df.loc[train_idx, TARGET]
    X_test = df.loc[test_idx, FEATURES]
    
    # آموزش مدل انسامبل
    ensemble = QuantileEnsemble()
    ensemble.fit(X_train, y_train)
    
    # پیش‌بینی
    ensemble_pred, individual_preds = ensemble.predict(X_test)
    
    # محاسبه واریانس برای اطمینان
    pred_array = np.array(individual_preds)
    ensemble_variance = np.var(pred_array, axis=0)
    
    # تنظیمات بازار
    context_df = df.loc[test_idx].copy()
    context_df = context_df.merge(historical_max_df, on='hour', how='left')
    
    adjusted_preds = smart_market_adjustment(ensemble_pred, context_df)
    
    # ذخیره نتایج
    df.loc[test_idx, "DECLARED"] = adjusted_preds
    df.loc[test_idx, "ENSEMBLE_VAR"] = ensemble_variance
    
    # محاسبه اطمینان (معکوس واریانس نرمال‌شده)
    if ensemble_variance.max() > 0:
        confidence = 1 - (ensemble_variance / ensemble_variance.max())
        df.loc[test_idx, "CONFIDENCE"] = confidence
    
    if day_idx % 30 == 0:
        print(f"پردازش روز {day_idx + 1} از {total_days}")

# =========================
# منطق نهایی بازار
# =========================
print("\nاعمال منطق نهایی بازار...")

# تنظیم برای ابراز = 0
if EBRAZ_COL in df.columns:
    df.loc[df[EBRAZ_COL] == 0, "DECLARED"] = 0

# اطمینان از مقادیر مثبت
df["DECLARED"] = df["DECLARED"].clip(lower=0)

# گرد کردن به عدد صحیح
df["DECLARED"] = df["DECLARED"].round(2)

# =========================
# ارزیابی
# =========================
print("\n" + "="*50)
print("ارزیابی نتایج:")
print("="*50)

# محاسبه خطا
err = df["DECLARED"] - df[TARGET]
valid_err = err[~np.isnan(err)]

if len(valid_err) > 0:
    # خطاهای مثبت (Over-prediction)
    over_err = valid_err[valid_err > 0]
    # خطاهای منفی (Under-prediction)
    under_err = valid_err[valid_err < 0]
    
    if len(over_err) > 0:
        mae_pos = np.mean(np.abs(over_err))
        mape_pos = np.mean(np.abs(over_err / df.loc[over_err.index, TARGET])) * 100
    else:
        mae_pos = 0
        mape_pos = 0
    
    if len(under_err) > 0:
        mae_neg = np.mean(np.abs(under_err))
        mape_neg = np.mean(np.abs(under_err / df.loc[under_err.index, TARGET])) * 100
    else:
        mae_neg = 0
        mape_neg = 0
    
    # محاسبه امتیاز بازار
    market_score = 5 * mae_pos + mae_neg
    
    # دقت کلی
    total_mae = np.mean(np.abs(valid_err))
    total_mape = np.mean(np.abs(valid_err / df.loc[valid_err.index, TARGET])) * 100
    
    print(f"MAE Positive (Over-prediction): {mae_pos:.2f} ({mape_pos:.1f}%)")
    print(f"MAE Negative (Under-prediction): {mae_neg:.2f} ({mape_neg:.1f}%)")
    print(f"Total MAE: {total_mae:.2f} ({total_mape:.1f}%)")
    print(f"Market Score (5*Over + Under): {market_score:.2f}")
    print(f"تعداد پیش‌بینی‌ها: {len(valid_err)}")
    
    # توزیع خطا
    print(f"\nتوزیع خطا:")
    print(f"  Over-prediction: {len(over_err)} ({len(over_err)/len(valid_err)*100:.1f}%)")
    print(f"  Under-prediction: {len(under_err)} ({len(under_err)/len(valid_err)*100:.1f}%)")
    print(f"  Exact: {sum(valid_err == 0)} ({sum(valid_err == 0)/len(valid_err)*100:.1f}%)")
else:
    print("هیچ داده معتبری برای ارزیابی وجود ندارد")

# =========================
# ویژگی‌های مهم
# =========================
if len(unique_days) > start_day:
    # آموزش مدل نهایی روی همه داده‌های آموزشی
    last_train_idx = df[DATE_COL].dt.date < unique_days[-1]
    X_final_train = df.loc[last_train_idx, FEATURES]
    y_final_train = df.loc[last_train_idx, TARGET]
    
    final_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        objective=quantile_objective(QUANTILE),
        random_state=42
    )
    final_model.fit(X_final_train, y_final_train)
    
    # اهمیت ویژگی‌ها
    feature_importance = pd.DataFrame({
        'feature': FEATURES,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n" + "="*50)
    print("۱۰ ویژگی مهم برتر:")
    print("="*50)
    for i, row in feature_importance.head(10).iterrows():
        print(f"{row['feature']}: {row['importance']:.4f}")

# =========================
# ذخیره نتایج
# =========================
print(f"\nذخیره نتایج در {OUTPUT_FILE}...")

# ایجاد DataFrame خلاصه نتایج
# اضافه کردن ستون‌های DATE_SHAMSI و ebraz از داده اصلی
results_summary = pd.DataFrame({
    'تاریخ میلادی': df[DATE_COL],
    'تاریخ شمسی': df['DATE_SHAMSI'] if 'DATE_SHAMSI' in df.columns else '',
    'ساعت': df[HOUR_COL],
    'قدرت واقعی': df[TARGET],
    'قدرت اعلان شده': df["DECLARED"],
    'خطا': err,
    'نوع خطا': pd.cut(err, 
                      bins=[-np.inf, -0.01, 0.01, np.inf], 
                      labels=['Under', 'Exact', 'Over']),
    'اطمینان': df["CONFIDENCE"],
    'واریانس انسامبل': df["ENSEMBLE_VAR"],
    'Lag_24': df["lag_24"],
    'importance_factor': df["importance_factor"],
    'practical': df["practical"],
    'ebraz': df[EBRAZ_COL] if EBRAZ_COL in df.columns else ''
})

# اضافه کردن ویژگی‌های مهم به عنوان ستون‌های اضافی
for feature in FEATURES[:5]:  # 5 ویژگی اول
    results_summary[feature] = df[feature]

# ذخیره در اکسل با چندین sheet
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    results_summary.to_excel(writer, sheet_name='نتایج', index=False)
    
    # sheet دوم: آمارهای خلاصه
    summary_stats = pd.DataFrame({
        'متریک': ['MAE Positive', 'MAE Negative', 'Total MAE', 'Market Score', 
                  'Over-prediction %', 'Under-prediction %', 'Exact %'],
        'مقدار': [f"{mae_pos:.2f}", f"{mae_neg:.2f}", f"{total_mae:.2f}", 
                  f"{market_score:.2f}", 
                  f"{len(over_err)/len(valid_err)*100:.1f}%" if len(valid_err) > 0 else "N/A",
                  f"{len(under_err)/len(valid_err)*100:.1f}%" if len(valid_err) > 0 else "N/A",
                  f"{sum(valid_err == 0)/len(valid_err)*100:.1f}%" if len(valid_err) > 0 else "N/A"]
    })
    summary_stats.to_excel(writer, sheet_name='آمار', index=False)
    
    # sheet سوم: اهمیت ویژگی‌ها
    if 'feature_importance' in locals():
        feature_importance.to_excel(writer, sheet_name='اهمیت ویژگی‌ها', index=False)

print("ذخیره‌سازی با موفقیت انجام شد!")
print(f"\nفایل خروجی: {OUTPUT_FILE}")
print("شامل سه Sheet:")
print("  1. نتایج: تمام پیش‌بینی‌ها و خطاها")
print("  2. آمار: خلاصه عملکرد مدل")
print("  3. اهمیت ویژگی‌ها: مهم‌ترین ویژگی‌های مدل")

خواندن داده...
ایجاد Lagها...
ایجاد ویژگی‌های پیشرفته...
حذف 26232 سطر با مقادیر NaN
تعداد ویژگی‌ها: 25

شروع پیش‌بینی Walk-Forward...
پردازش روز 31 از 550
پردازش روز 61 از 550
پردازش روز 91 از 550
پردازش روز 121 از 550
پردازش روز 151 از 550
پردازش روز 181 از 550
پردازش روز 211 از 550
پردازش روز 241 از 550
پردازش روز 271 از 550
پردازش روز 301 از 550
پردازش روز 331 از 550
پردازش روز 361 از 550
پردازش روز 391 از 550
پردازش روز 421 از 550
پردازش روز 451 از 550
پردازش روز 481 از 550
پردازش روز 511 از 550
پردازش روز 541 از 550

اعمال منطق نهایی بازار...

ارزیابی نتایج:
MAE Positive (Over-prediction): 11.12 (inf%)
MAE Negative (Under-prediction): 12.39 (10.7%)
Total MAE: 9.70 (inf%)
Market Score (5*Over + Under): 67.98
تعداد پیش‌بینی‌ها: 12456

توزیع خطا:
  Over-prediction: 5226 (42.0%)
  Under-prediction: 5066 (40.7%)
  Exact: 2164 (17.4%)

۱۰ ویژگی مهم برتر:
MA_24: 0.7477
ratio_24_48: 0.1160
MA_168: 0.0251
hour: 0.0229
month_cos: 0.0196
delta_48: 0.0191
ROTOOBAT: 0.0186
lag24_hour: 0.0139
